# 02 — Matrix unit simulation: naive vs tiled matmul

**Learning goal.** Build intuition for why TPUs are matmul machines, by measuring the effect of **tiling** on a CPU. The same idea — operate on small dense tiles that fit in fast memory — is what the TPU's Matrix Unit (MXU) is built around.

**What you'll observe.**
- A naive triple-loop matmul (very slow).
- The same multiplication using a **tiled** block matmul that walks the inputs in cache-friendly chunks.
- NumPy's `@` operator, which dispatches to BLAS (the same algorithmic family as tiling, plus SIMD).

**Knobs to tune.**
- `N` — matrix size. The cache effect grows with `N`.
- `TILE` — tile size. Too small = overhead dominates. Too big = doesn't fit in L1/L2.

**Failure modes to watch.**
- Running with very large `N` will exhaust CPU memory and make the naive loop take minutes. Keep `N <= 256` for naive; NumPy can go bigger.
- Timing is noisy on a busy laptop. Run cells more than once if numbers look wild.

In [ ]:
import sys, os
from pathlib import Path
# Add the parent of cloud_tpu_lab to sys.path so `cloud_tpu_lab.src.*` imports work.
HERE = Path(os.getcwd()).resolve()
_root = HERE
for _ in range(5):
    if (_root / "cloud_tpu_lab").exists():
        break
    _root = _root.parent
sys.path.insert(0, str(_root))
sys.path.insert(0, "..")
PLOT_DIR = Path("../artifacts/plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)
print("repo root:", _root)
print("plot dir:", PLOT_DIR.resolve())

## Naive matmul (triple loop)

This is the textbook form. It has poor cache locality: every iteration of the inner loop touches a new cache line of `B`.

In [ ]:
import numpy as np
import time

def matmul_naive(A, B):
    n, k = A.shape
    k2, m = B.shape
    assert k == k2
    C = np.zeros((n, m), dtype=A.dtype)
    for i in range(n):
        for j in range(m):
            s = 0.0
            for p in range(k):
                s += A[i, p] * B[p, j]
            C[i, j] = s
    return C

N = 96  # keep small — naive triple-loop is O(N^3) and pure Python
rng = np.random.default_rng(0)
A = rng.standard_normal((N, N)).astype(np.float32)
B = rng.standard_normal((N, N)).astype(np.float32)

t0 = time.perf_counter()
C_naive = matmul_naive(A, B)
naive_s = time.perf_counter() - t0
print(f"naive matmul N={N}  took {naive_s*1000:.1f} ms")

## Tiled (blocked) matmul

We walk the output in `TILE x TILE` blocks and accumulate them. Each inner block fits in cache, so we reuse the loaded values many times before evicting them. On the TPU's MXU the equivalent idea is built in hardware: it operates on a fixed-size systolic tile and streams operands through it.

In [ ]:
def matmul_tiled(A, B, tile=32):
    n, k = A.shape
    _, m = B.shape
    C = np.zeros((n, m), dtype=A.dtype)
    for ii in range(0, n, tile):
        for jj in range(0, m, tile):
            for pp in range(0, k, tile):
                # block-level matmul using NumPy (already cache-friendly)
                C[ii:ii+tile, jj:jj+tile] += A[ii:ii+tile, pp:pp+tile] @ B[pp:pp+tile, jj:jj+tile]
    return C

t0 = time.perf_counter()
C_tiled = matmul_tiled(A, B, tile=32)
tiled_s = time.perf_counter() - t0
print(f"tiled matmul N={N}  took {tiled_s*1000:.1f} ms")

# Sanity check
print("max abs error vs naive:", float(np.max(np.abs(C_tiled - C_naive))))

## NumPy reference (BLAS)

NumPy's `@` operator dispatches to a BLAS library (MKL/OpenBLAS) which uses tiling + SIMD + multi-threading. This is what a CPU can do when nothing is left on the table.

In [ ]:
t0 = time.perf_counter()
C_blas = A @ B
blas_s = time.perf_counter() - t0
print(f"BLAS  matmul N={N}  took {blas_s*1000:.3f} ms")
print("max abs error vs naive:", float(np.max(np.abs(C_blas - C_naive))))

## Plot: time vs implementation

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

labels = ["naive (python loop)", "tiled (blocks of 32)", "BLAS (np.matmul)"]
times = [naive_s, tiled_s, blas_s]

fig = plt.figure(figsize=(7, 4))
plt.bar(labels, times)
plt.ylabel("time (s)")
plt.yscale("log")
plt.title(f"Matrix multiply N={N}, fp32 (log scale)")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
out_png = PLOT_DIR / "nb02_matmul_compare.png"
plt.savefig(out_png, dpi=120)
plt.show()
print("saved:", out_png)

## Why this matters for TPUs

A TPU's **Matrix Unit (MXU)** is a hardware tile that multiplies a fixed-size block per cycle (think 128x128). The compiler's job is to slice your matmul into MXU-sized tiles and stream operands so the unit never starves.

- If your problem decomposes cleanly into MXU tiles, you get near-peak FLOPS.
- If shapes don't divide cleanly, the MXU runs with padding and effective throughput drops.
- If your operation isn't a matmul at all, the MXU doesn't help — vector/scalar units handle elementwise ops at a much lower peak rate.

This is also why you'll see advice like "make your hidden dim a multiple of 128". It maps directly onto MXU tile sizes.

*(No private microarchitecture details here — just the publicly documented idea.)*

## Exercises

1. Sweep `tile` over `[8, 16, 32, 64, 128]` in the tiled function. Where is the minimum? Why does very small tile not help, and very large tile not help either?
2. Change `N` to 192 and 256 and re-run all three. Does the gap between naive and tiled widen or narrow?
3. Why is the BLAS time roughly flat as `N` grows compared to the naive loop?
4. Time `A.T @ B` vs `A @ B`. Is the transpose free? Why might it not be?
5. Run the tiled multiply with `dtype=np.float16` (or `bfloat16` via `ml_dtypes` if installed). Does it get faster on CPU? On a TPU it would — explain why.